# Data Preparation: Olist Order Reviews

Structure follows the validation session: **data cleaning, data transformation, feature engineering, dimensionality reduction**.

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raw = pd.read_csv("olist_order_reviews_dataset.csv")
print("Shape:", raw.shape)
raw.info()


print("*********************************************************************")
print(raw.describe())
print("*********************************************************************")
print(raw.info())
print("*********************************************************************")
raw

Shape: (99224, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB
*********************************************************************
       review_score
count  99224.000000
mean       4.086421
std        1.347579
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000
*********************************************************************
<class 'pandas.core.frame.D

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


## 1. Data cleaning

### 1.1 Missing values and duplicates



In [26]:
print(raw.isna().sum())
print("\nDuplicate review_id:", raw["review_id"].duplicated().sum())
print("Extra reviews on the same order:", raw["order_id"].duplicated().sum())

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Duplicate review_id: 814
Extra reviews on the same order: 551


### 1.2 Validate review score



In [27]:
invalid = ~raw["review_score"].between(1, 5)
print("Invalid scores removed:", invalid.sum())

rev = raw[~invalid].copy()

Invalid scores removed: 0


### 1.3 Fix data types

Dates are read as text, so they are converted to datetime before any date check or calculation.

In [28]:
for col in ["review_creation_date", "review_answer_timestamp"]:
    rev[col] = pd.to_datetime(rev[col])

rev.dtypes

,0
review_id,object
order_id,object
review_score,int64
review_comment_title,object
review_comment_message,object
review_creation_date,datetime64[ns]
review_answer_timestamp,datetime64[ns]


### 1.4 Date sanity checks



In [29]:
rev["flag_answer_before_review"] = rev["review_answer_timestamp"] < rev["review_creation_date"]
print("Answers before review (flagged):", rev["flag_answer_before_review"].sum())

Answers before review (flagged): 0


## 2. Data transformation

### 2.1 Reduce to one review per order

**bold text**

In [30]:
rows_before = len(rev)

rev = (rev.sort_values("review_creation_date")
          .drop_duplicates("order_id", keep="last")
          .reset_index(drop=True))

print("Rows before:", rows_before)
print("Rows after:", len(rev))
print("Rows dropped:", rows_before - len(rev))
assert rev["order_id"].is_unique

Rows before: 99224
Rows after: 98673
Rows dropped: 551


## 3. Feature engineering

### 3.1 Comment presence

The two optional text fields are replaced by one boolean, `has_comment`. It is True if either field has a value. The Portuguese text is removed here because the current models do not use it.

In [31]:
rev["has_comment"] = rev[["review_comment_title", "review_comment_message"]].notna().any(axis=1)
rev = rev.drop(columns=["review_comment_title", "review_comment_message"])

print(rev["has_comment"].value_counts())

rev

has_comment
False    56171
True     42502
Name: count, dtype: int64


,review_id,order_id,review_score,review_creation_date,review_answer_timestamp,flag_answer_before_review,has_comment
0,69ac6a27fde9855ebeaaecac0f78058b,809a282bbd5dbcabb6f2f724fca862ec,1,2016-10-02,2016-10-26 12:31:00,False,True
1,6916ca4502d6d3bfd39818759d55d536,bfbd0f9bdef84302105ad712db648a6c,1,2016-10-06,2016-10-07 18:32:28,False,True
2,49f695dffa457eaba90d388a5c37e942,e5215415bb6f76fe3b7cb68103a0d1c0,1,2016-10-09,2016-10-11 14:31:29,False,True
3,743d98b1a4782f0646898fc915ef002a,e2144124f98f3bf46939bc5183104041,4,2016-10-15,2016-10-16 03:20:17,False,False
4,b2d5d8db2a841d27a72e4c06c6212368,9aa3197e4887919fde0307fc23601d7a,4,2016-10-15,2016-10-17 21:02:49,False,True
...,...,...,...,...,...,...,...
98668,12588003a92c76aefda4b34695db04ce,576466ecdb1ad73bb848943a98649668,5,2018-08-31,2018-09-02 18:17:38,False,False
98669,efc9adf3e8945a6236b399ddea03a1f5,9344079d5fbb7dd79af908b7eb667db7,4,2018-08-31,2018-08-31 23:58:00,False,False
98670,2c573bdf789cae8761dc8357cc6d2663,919baca007d9525b6668c18f79a33197,1,2018-08-31,2018-08-31 15:47:56,False,True
98671,0de5afcfb89bbc4929292053e2950c15,5482a1c74b1b5fb1af78a425b7ceabb8,3,2018-08-31,2018-08-31 21:31:40,False,True


### 3.2 Response time

`days_to_answer` is the gap between the review and the seller's answer, in fractional days. Using `.dt.days` would truncate any gap under one day to 0, so the total seconds are divided by 86,400 instead.

In [32]:
rev["days_to_answer"] = (
    (rev["review_answer_timestamp"] - rev["review_creation_date"]).dt.total_seconds() / 86400
)

print(rev["days_to_answer"].describe())

rev

count    98673.000000
mean         3.149600
std          9.909520
min          0.089225
25%          1.004468
50%          1.674931
75%          3.102060
max        518.699213
Name: days_to_answer, dtype: float64


,review_id,order_id,review_score,review_creation_date,review_answer_timestamp,flag_answer_before_review,has_comment,days_to_answer
0,69ac6a27fde9855ebeaaecac0f78058b,809a282bbd5dbcabb6f2f724fca862ec,1,2016-10-02,2016-10-26 12:31:00,False,True,24.521528
1,6916ca4502d6d3bfd39818759d55d536,bfbd0f9bdef84302105ad712db648a6c,1,2016-10-06,2016-10-07 18:32:28,False,True,1.772546
2,49f695dffa457eaba90d388a5c37e942,e5215415bb6f76fe3b7cb68103a0d1c0,1,2016-10-09,2016-10-11 14:31:29,False,True,2.605197
3,743d98b1a4782f0646898fc915ef002a,e2144124f98f3bf46939bc5183104041,4,2016-10-15,2016-10-16 03:20:17,False,False,1.139086
4,b2d5d8db2a841d27a72e4c06c6212368,9aa3197e4887919fde0307fc23601d7a,4,2016-10-15,2016-10-17 21:02:49,False,True,2.876956
...,...,...,...,...,...,...,...,...
98668,12588003a92c76aefda4b34695db04ce,576466ecdb1ad73bb848943a98649668,5,2018-08-31,2018-09-02 18:17:38,False,False,2.762245
98669,efc9adf3e8945a6236b399ddea03a1f5,9344079d5fbb7dd79af908b7eb667db7,4,2018-08-31,2018-08-31 23:58:00,False,False,0.998611
98670,2c573bdf789cae8761dc8357cc6d2663,919baca007d9525b6668c18f79a33197,1,2018-08-31,2018-08-31 15:47:56,False,True,0.658287
98671,0de5afcfb89bbc4929292053e2950c15,5482a1c74b1b5fb1af78a425b7ceabb8,3,2018-08-31,2018-08-31 21:31:40,False,True,0.896991


### 3.3 Low review flag

`is_low_review` marks scores of 2 or below.

In [37]:
rev["is_low_review"] = rev["review_score"] <= 2

print("Share of low reviews:", round(rev["is_low_review"].mean(), 3))
rev

Share of low reviews: 0.147


,review_id,order_id,review_score,review_creation_date,review_answer_timestamp,flag_answer_before_review,has_comment,days_to_answer,is_low_review
0,69ac6a27fde9855ebeaaecac0f78058b,809a282bbd5dbcabb6f2f724fca862ec,1,2016-10-02,2016-10-26 12:31:00,False,True,24.521528,True
1,6916ca4502d6d3bfd39818759d55d536,bfbd0f9bdef84302105ad712db648a6c,1,2016-10-06,2016-10-07 18:32:28,False,True,1.772546,True
2,49f695dffa457eaba90d388a5c37e942,e5215415bb6f76fe3b7cb68103a0d1c0,1,2016-10-09,2016-10-11 14:31:29,False,True,2.605197,True
3,743d98b1a4782f0646898fc915ef002a,e2144124f98f3bf46939bc5183104041,4,2016-10-15,2016-10-16 03:20:17,False,False,1.139086,False
4,b2d5d8db2a841d27a72e4c06c6212368,9aa3197e4887919fde0307fc23601d7a,4,2016-10-15,2016-10-17 21:02:49,False,True,2.876956,False
...,...,...,...,...,...,...,...,...,...
98668,12588003a92c76aefda4b34695db04ce,576466ecdb1ad73bb848943a98649668,5,2018-08-31,2018-09-02 18:17:38,False,False,2.762245,False
98669,efc9adf3e8945a6236b399ddea03a1f5,9344079d5fbb7dd79af908b7eb667db7,4,2018-08-31,2018-08-31 23:58:00,False,False,0.998611,False
98670,2c573bdf789cae8761dc8357cc6d2663,919baca007d9525b6668c18f79a33197,1,2018-08-31,2018-08-31 15:47:56,False,True,0.658287,True
98671,0de5afcfb89bbc4929292053e2950c15,5482a1c74b1b5fb1af78a425b7ceabb8,3,2018-08-31,2018-08-31 21:31:40,False,True,0.896991,False


## 4. Dimensionality reduction


In [34]:
num_cols = ["review_score", "days_to_answer", "has_comment", "is_low_review"]
print(rev[num_cols].astype(float).corr().round(2))

                review_score  days_to_answer  has_comment  is_low_review
review_score            1.00            0.01        -0.25          -0.88
days_to_answer          0.01            1.00         0.00          -0.00
has_comment            -0.25            0.00         1.00           0.27
is_low_review          -0.88           -0.00         0.27           1.00


## Output



In [35]:
rev.to_parquet("clean_order_reviews.parquet", index=False)
print("Saved:", rev.shape)

Saved: (98673, 9)
